# Cross-task patching — aggregation-head OV transplant
`experiments/interplay/08_crosstask_agg_ov.ipynb`

Transplant the **aggregation heads' OV write** (their `hook_z` contribution) from a DONOR
prompt's **final/query position** into a SOURCE prompt's final position, then greedy-decode.
Aggregation heads attend final→output and write at the final position, so this installs the
donor's *integrated* task representation — the most task-vector-like object — and nothing else.
Single position ⇒ no alignment.

Built on the repo's real API (TransformerLens `HookedTransformer`, hooks on
`blocks.{L}.attn.hook_z` shaped `(batch, seq, n_heads, d_head)`, `check_correct_multitoken`).

**Bins** (exact full-string match, priority order):
`acc_src` = source_rule(src_input) · `acc_donor` = donor_rule(src_input) ·
`acc_leak` = donor prompt's stored answer · `other` = none.

Full source×donor matrix per family, N=10. Diagonal = self-transplant control (donor_rule==source_rule
there ⇒ scores as `acc_src`; should stay high if the patch is non-destructive).

In [1]:
import sys, itertools
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd, torch
from tqdm.auto import tqdm

# repo root on path (notebook lives in experiments/interplay/)
sys.path.insert(0, str(Path('../..').resolve()))
import experiments.pairing._common as C          # load_model, load_scope_heads, RESULTS_DIR, save_results, save_fig
from data.loaders import load_dataset
from data.tasks import _NONCE_TRANSFORMS, _ARITH_SPECS, NONCE_TASKS, ARITH_TASKS
from utils.eval import check_correct_multitoken
from utils.positions import find_per_demo_positions_robust

DATASET  = 'nonce+arithmetic'
HEAD_PCT = 10
SCOPE    = 'pooled'
N        = 10
MAX_NEW  = 20

model  = C.load_model()                            # HookedTransformer, fp16, cuda
n_heads, d_head = model.cfg.n_heads, model.cfg.d_head
splits = load_dataset(DATASET)

# aggregation heads (requires score_heads.py to have produced the head-set cache)
agg_heads, agg_rand, _ms = C.load_scope_heads(DATASET, HEAD_PCT, SCOPE, 'aggregation')
print('aggregation heads:', sorted(agg_heads))

def apply_rule(task, x):
    if task in _NONCE_TRANSFORMS:
        return _NONCE_TRANSFORMS[task](x)
    fn, _lo, _hi = _ARITH_SPECS[task]
    return str(fn(int(x)))

nonce = [t for t in splits if t in set(NONCE_TASKS)]
arith = [t for t in splits if t in set(ARITH_TASKS)]
print('nonce', len(nonce), '| arith', len(arith))


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-3.2-3B.
401 Client Error. (Request ID: Root=1-6a301c57-412403662f9a68292ad09f67;1594d9b5-a9d8-499f-af87-cb0de15e3995)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-3.2-3B/resolve/main/config.json.
Access to model meta-llama/Llama-3.2-3B is restricted. You must have access to it and be authenticated to access it. Please log in.

In [ ]:
agg_by_layer = defaultdict(list)
for L, h in agg_heads:
    agg_by_layer[L].append(h)
agg_by_layer = dict(agg_by_layer)

@torch.no_grad()
def capture_agg_z(prompt):
    # Donor aggregation-head z at the final position: {(L,h): tensor(d_head)}
    toks = model.to_tokens(prompt, prepend_bos=True)
    last = toks.shape[1] - 1
    zfilter = lambda n: 'hook_z' in n
    _, cache = model.run_with_cache(toks, names_filter=zfilter)
    store = {}
    for L, hs in agg_by_layer.items():
        z = cache[f'blocks.{L}.attn.hook_z'][0, last]        # (n_heads, d_head)
        for h in hs:
            store[(L, h)] = z[h].detach().clone()
    del cache
    return store

def make_patch_hooks(donor_store, src_last):
    # fwd_hooks overwriting aggregation z at the source final position.
    # During generation seq grows; we always target the original final position only.
    hooks = []
    for L, hs in agg_by_layer.items():
        def hook(z, hook, _hs=hs, _L=L):
            # z: (batch, seq, n_heads, d_head)
            if z.shape[1] > src_last:
                for h in _hs:
                    z[0, src_last, h, :] = donor_store[(_L, h)].to(z.dtype)
            return z
        hooks.append((f'blocks.{L}.attn.hook_z', hook))
    return hooks

@torch.no_grad()
def patched_decode(src_prompt, donor_store):
    toks = model.to_tokens(src_prompt, prepend_bos=True)
    src_last = toks.shape[1] - 1
    hooks = make_patch_hooks(donor_store, src_last)
    gen = []
    cur = toks.clone()
    for _ in range(MAX_NEW):
        logits = model.run_with_hooks(cur, fwd_hooks=hooks)[0, -1]
        nt = logits.argmax().item()
        gen.append(nt)
        cur = torch.cat([cur, torch.tensor([[nt]], device=cur.device)], dim=1)
    return model.tokenizer.decode(gen).strip()

def classify(dec, s_tgt, d_tgt, l_tgt):
    dec = dec.strip()
    if dec == s_tgt.strip(): return 'acc_src'
    if dec == d_tgt.strip(): return 'acc_donor'
    if dec == l_tgt.strip(): return 'acc_leak'
    return 'other'


In [ ]:
rows=[]
for family, tasks in [('nonce', nonce), ('arith', arith)]:
    # donor z cached per task (each donor task reused across all source rows)
    dcache = {t: [capture_agg_z(p['prompt']) for p in splits[t]['icl_prompts'][:N]]
              for t in tasks}
    for src_t in tqdm(tasks, desc=family):
        src_ps = splits[src_t]['icl_prompts'][:N]
        for dnr_t in tasks:                              # full matrix incl. diagonal
            dnr_ps = splits[dnr_t]['icl_prompts'][:N]
            c = dict(acc_src=0, acc_donor=0, acc_leak=0, other=0)
            for j, sp in enumerate(src_ps):
                dec   = patched_decode(sp['prompt'], dcache[dnr_t][j])
                s_tgt = apply_rule(src_t, sp['query_input'])      # == sp['query_output']
                d_tgt = apply_rule(dnr_t, sp['query_input'])
                l_tgt = dnr_ps[j]['query_output']
                c[classify(dec, s_tgt, d_tgt, l_tgt)] += 1
            rows.append(dict(family=family, src=src_t, donor=dnr_t,
                             **{k: v/N for k, v in c.items()}))
R = pd.DataFrame(rows)
C.save_results('08_crosstask_agg_ov', {'df': R})
print(R.groupby('family')[['acc_src','acc_donor','acc_leak','other']].mean().round(3))


In [ ]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt, seaborn as sns

BINS=['acc_donor','acc_src','acc_leak','other']
TITLES={'acc_donor':'transplanted (donor) task','acc_src':'original (source) task',
        'acc_leak':'donor answer leaked','other':'broken / other'}
for family in ['nonce','arith']:
    sub=R[R.family==family]; order=sorted(sub['src'].unique())
    fig,axes=plt.subplots(1,4,figsize=(22,5))
    for ax,b in zip(axes,BINS):
        mat=sub.pivot(index='src',columns='donor',values=b).reindex(index=order,columns=order)
        sns.heatmap(mat,ax=ax,vmin=0,vmax=1,cmap='viridis',annot=True,fmt='.2f',
                    annot_kws={'size':7},cbar_kws={'shrink':.7})
        ax.set(title=TITLES[b],xlabel='donor',ylabel='source')
    fig.suptitle(f'{family}: aggregation-head OV transplant @ final position, N={N}',y=1.02)
    plt.tight_layout()
    C.save_fig(fig, f'08_crosstask_agg_ov_{family}')
    plt.show()
print('saved heatmaps to', C.RESULTS_DIR)


## Read it
Rows = **source** prompt, columns = **donor** (transplanted) task.

- **`transplanted (donor)` bright off-diagonal** → the aggregation OV write is a transplantable
  task code: it installs the donor task on the source query.
- **`original (source)` bright on the diagonal, dark off-diagonal** → the patch reliably replaces
  the source task (necessary); diagonal confirms a same-task donor is non-destructive — check this first.
- **`donor answer leaked` bright** → write carries answer *content*, not a reusable mapping
  (task-recognition, not task-learning).
- **nonce vs arith**: compare the donor heatmaps — transfer in one family but not the other is the
  mechanism boundary. Block structure = task-similarity in the aggregated code.

Sanity before trusting nulls: print one `dec` vs an unpatched `check_correct_multitoken` decode, and
confirm the `original` diagonal is high. Control: rerun with `agg_rand` in place of `agg_heads` to
show transfer is aggregation-specific, not any-K-head perturbation.